# Results analysis

Reads the result CSVs in `results/tables`, prints the results tables and saves the figures to `results/plots`.

Only runs with the main settings are used (10 clients, 1 local epoch, lr 0.003 for the hybrid model and 0.001 for the classical CNN), so the tuning and test runs are skipped.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

%matplotlib inline

In [ ]:
TABLES_DIR = "../results/tables"
PLOTS_DIR  = "../results/plots"

CLIENTS      = 10
EPOCHS       = 1
LR           = 0.003
CLASSICAL_LR = 0.001
MU           = 0.01
TARGET       = 75
MAIN_SPLITS  = ["IID", "Dirichlet α=0.5", "Dirichlet α=0.1"]

PATTERN = re.compile(r"^(classical|qfl_fedavg|qfl_fedprox_full|qfl_fedprox|qfl_weighted)_(iid|dirichlet)_a([\d.]+)"
                     r"(?:_mu([\d.e-]+))?(?:_lam([\d.]+))?_c(\d+)_e(\d+)_lr([\d.e-]+)_s(\d+)\.csv$")

NAMES = {
    "qfl_fedavg":       "QFL FedAvg",
    "qfl_fedprox":      "Quantum FedProx",
    "qfl_fedprox_full": "Full FedProx",
    "qfl_weighted":     "Weighted Agg",
    "classical":        "Classical FedAvg",
}
ORDER  = ["QFL FedAvg", "Quantum FedProx", "Full FedProx", "Weighted Agg", "Classical FedAvg"]
COLORS = {"QFL FedAvg": "#7f7f7f", "Quantum FedProx": "#d62728", "Full FedProx": "#ff7f0e",
          "Weighted Agg": "#1f77b4", "Classical FedAvg": "#2ca02c"}

os.makedirs(PLOTS_DIR, exist_ok=True)

## Load the runs

In [ ]:
rows = []
for fname in sorted(os.listdir(TABLES_DIR)):
    match = PATTERN.match(fname)
    if not match:
        continue
    kind, partition, alpha, mu, lam, clients, epochs, lr, seed = match.groups()
    want_lr = CLASSICAL_LR if kind == "classical" else LR
    if int(clients) != CLIENTS or int(epochs) != EPOCHS or float(lr) != want_lr:
        continue

    df = pd.read_csv(os.path.join(TABLES_DIR, fname))
    acc = df["accuracy"] * 100
    rows.append({
        "strategy": NAMES[kind],
        "split": "IID" if partition == "iid" else f"Dirichlet α={float(alpha):g}",
        "mu": float(mu) if mu else np.nan,
        "seed": int(seed),
        "emd": df["emd"].iloc[0],
        "rounds": len(df),
        "final": acc.iloc[-1],
        "last5": acc.iloc[-5:].mean(),
        "best": acc.max(),
        "sec_per_round": df["time"].mean(),
        "curve": acc.tolist(),
    })

runs = pd.DataFrame(rows)
main = runs[runs["mu"].isna() | (runs["mu"] == MU)]
splits = main.groupby("split")["emd"].mean().sort_values().index.tolist()
strategies = [s for s in ORDER if s in set(main["strategy"])]
print(f"Loaded {len(runs)} runs ({len(main)} in the main comparison)")

## Results table

Test accuracy (%) averaged over the last 5 rounds. When a setting has more than one seed the cell shows mean ± std over seeds, `-` means that run was not done.

In [ ]:
summary = (main.groupby(["split", "strategy"])
           .agg(emd=("emd", "mean"), seeds=("seed", "nunique"), last5=("last5", "mean"),
                last5_std=("last5", "std"), final=("final", "mean"), best=("best", "mean"),
                sec_per_round=("sec_per_round", "mean"))
           .reset_index())
summary["last5_std"] = summary["last5_std"].fillna(0)
summary["split_order"] = summary["split"].map({s: i for i, s in enumerate(splits)})
summary["strategy_order"] = summary["strategy"].map({s: i for i, s in enumerate(ORDER)})
summary = (summary.sort_values(["split_order", "strategy_order"])
           .drop(columns=["split_order", "strategy_order"]).reset_index(drop=True))
summary.to_csv(os.path.join(TABLES_DIR, "summary.csv"), index=False)

print("| Split | EMD | " + " | ".join(strategies) + " |")
print("|---" * (len(strategies) + 2) + "|")
for split in splits:
    sub = summary[summary["split"] == split].set_index("strategy")
    cells = []
    for strategy in strategies:
        if strategy not in sub.index:
            cells.append("-")
        elif sub.loc[strategy, "seeds"] > 1:
            cells.append(f"{sub.loc[strategy, 'last5']:.2f} ± {sub.loc[strategy, 'last5_std']:.2f}")
        else:
            cells.append(f"{sub.loc[strategy, 'last5']:.2f}")
    print(f"| {split} | {sub['emd'].mean():.3f} | " + " | ".join(cells) + " |")

Every run on its own (last 5 rounds mean, one column per seed):

In [ ]:
per_seed = main.pivot_table(index=["split", "strategy"], columns="seed", values="last5")
order = [(sp, st) for sp in splits for st in strategies if (sp, st) in per_seed.index]
per_seed.loc[order].round(2).fillna("")

Rounds until the mean accuracy curve (over seeds) first reaches the target, blank means it never did:

In [ ]:
reach = {}
for (split, strategy), sub in main.groupby(["split", "strategy"]):
    n = min(len(c) for c in sub["curve"])
    mean_curve = np.mean([c[:n] for c in sub["curve"]], axis=0)
    hit = np.where(mean_curve >= TARGET)[0]
    reach[(split, strategy)] = hit[0] + 1 if len(hit) else np.nan

print(f"Target = {TARGET:g}% test accuracy")
pd.Series(reach).unstack().reindex(index=splits, columns=strategies).fillna("")

## Accuracy per round

Mean over seeds, the shaded band is ±1 std.

In [ ]:
fig, axes = plt.subplots(1, len(MAIN_SPLITS), figsize=(5 * len(MAIN_SPLITS), 4.2), sharey=True)
axes = np.atleast_1d(axes)
for ax, split in zip(axes, MAIN_SPLITS):
    for strategy in ORDER:
        sub = main[(main["split"] == split) & (main["strategy"] == strategy)]
        if sub.empty:
            continue
        n = min(len(c) for c in sub["curve"])
        curves = np.array([c[:n] for c in sub["curve"]])
        mean = curves.mean(axis=0)
        rounds = np.arange(1, n + 1)
        ax.plot(rounds, mean, label=strategy, color=COLORS[strategy], lw=2)
        if len(curves) > 1:
            std = curves.std(axis=0, ddof=1)
            ax.fill_between(rounds, mean - std, mean + std, color=COLORS[strategy], alpha=0.15)
    emd = main[main["split"] == split]["emd"].mean()
    ax.set_title(f"{split} (EMD {emd:.2f})", fontsize=11)
    ax.set_xlabel("Communication round")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_ylim(0, 100)
    ax.grid(True, ls="--", alpha=0.4)
axes[0].set_ylabel("Test accuracy (%)")
axes[0].legend(fontsize=8, loc="lower right")
fig.tight_layout()

out = os.path.join(PLOTS_DIR, "accuracy_curves.png")
fig.savefig(out, dpi=200)
plt.show()
print(f"Saved: {out}")

## Accuracy vs measured non-IID level

Each point is one run. The x axis is the EMD proxy measured on that run's client split (0 = every client has the same label mix, 2 = no two clients share a class).

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
for strategy in strategies:
    sub = main[main["strategy"] == strategy]
    ax.scatter(sub["emd"], sub["last5"], label=strategy, color=COLORS[strategy], s=28, alpha=0.8)
ax.set_xlabel("Measured non-IID level (EMD proxy)")
ax.set_ylabel("Test accuracy, mean of last 5 rounds (%)")
ax.set_ylim(0, 100)
ax.grid(True, ls="--", alpha=0.4)
ax.legend(fontsize=8, loc="lower left")
fig.tight_layout()

out = os.path.join(PLOTS_DIR, "accuracy_vs_emd.png")
fig.savefig(out, dpi=200)
plt.show()
print(f"Saved: {out}")

## Strategy comparison

In [ ]:
width = 0.8 / len(strategies)
x = np.arange(len(MAIN_SPLITS))
fig, ax = plt.subplots(figsize=(8, 4.5))
for i, strategy in enumerate(strategies):
    sub = summary[summary["strategy"] == strategy].set_index("split").reindex(MAIN_SPLITS)
    ax.bar(x - 0.4 + width * (i + 0.5), sub["last5"], width, yerr=sub["last5_std"],
           label=strategy, color=COLORS[strategy], capsize=3)
ax.set_xticks(x)
ax.set_xticklabels(MAIN_SPLITS)
ax.set_ylabel("Test accuracy, mean of last 5 rounds (%)")
ax.set_ylim(0, 100)
ax.grid(axis="y", ls="--", alpha=0.4)
ax.legend(fontsize=8, ncol=len(strategies), loc="upper center")
fig.tight_layout()

out = os.path.join(PLOTS_DIR, "strategy_comparison.png")
fig.savefig(out, dpi=200)
plt.show()
print(f"Saved: {out}")

## Proximal strength sweep

Quantum FedProx vs Full FedProx at the strongest skew for different μ. Only seeds that were run for every μ are used, so all points are compared on the same client splits. The dashed line is plain QFL FedAvg (μ = 0) on the same seeds.

In [ ]:
prox = runs[runs["strategy"].isin(["Quantum FedProx", "Full FedProx"])]
split = prox.groupby("split")["emd"].mean().idxmax()
prox = prox[prox["split"] == split]
common = set.intersection(*prox.groupby(["strategy", "mu"])["seed"].apply(set))
prox = prox[prox["seed"].isin(common)]

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for strategy in ["Quantum FedProx", "Full FedProx"]:
    g = prox[prox["strategy"] == strategy].groupby("mu")["last5"]
    ax.errorbar(g.mean().index, g.mean().values, yerr=g.std().fillna(0).values, label=strategy,
                color=COLORS[strategy], marker="o", lw=2, capsize=3)
base = runs[(runs["strategy"] == "QFL FedAvg") & (runs["split"] == split) & runs["seed"].isin(common)]["last5"]
ax.axhline(base.mean(), color=COLORS["QFL FedAvg"], ls="--", label="QFL FedAvg (μ = 0)")
ax.set_xscale("log")
ax.set_xlabel("Proximal strength μ")
ax.set_ylabel("Test accuracy, mean of last 5 rounds (%)")
ax.set_title(f"{split}, seeds {sorted(common)}", fontsize=11)
ax.grid(True, ls="--", alpha=0.4)
ax.legend(fontsize=8)
fig.tight_layout()

out = os.path.join(PLOTS_DIR, "mu_sweep.png")
fig.savefig(out, dpi=200)
plt.show()
print(f"Saved: {out}")

prox.pivot_table(index="mu", columns="strategy", values="last5").round(2)